# Stickman Hybrid Studio — Kaggle GPU Worker (T4 x2)
1. Set **Accelerator**: `GPU T4 x2` & **Internet**: `On`.
2. Run Cell 1 once to install dependencies.
3. Run Cell 2 to kill any old processes, pull the latest `worker_server.py`, and print your **Cloudflare Tunnel URL**.

In [ ]:
!pip install -q --upgrade diffusers transformers accelerate sentencepiece protobuf fastapi uvicorn pillow
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
print('Setup complete!')

In [ ]:
import os, re, subprocess, time, urllib.request

# Kill any leftover uvicorn / cloudflared from previous runs so VRAM & port 8000 are clean
subprocess.run(['pkill', '-9', '-f', 'uvicorn'], check=False)
subprocess.run(['pkill', '-9', '-f', 'cloudflared'], check=False)
time.sleep(1)

# Always pull latest worker_server.py from GitHub (cache-busted)
url = f'https://raw.githubusercontent.com/lukman582890-wq/stickman_project/main/kaggle/worker_server.py?t={int(time.time())}'
urllib.request.urlretrieve(url, 'worker_server.py')
print('Downloaded latest worker_server.py')

os.environ['STICKMAN_API_KEY'] = ''
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

uvicorn_proc = subprocess.Popen(['uvicorn', 'worker_server:app', '--host', '0.0.0.0', '--port', '8000'])
time.sleep(3)
cf_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)
for line in cf_proc.stdout:
    m = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if m:
        print('\n========================================')
        print('KAGGLE WORKER URL:', m.group(0))
        print('========================================\n')
        break
cf_proc.wait()